# Общая CNN для датасета МРТ

Датасет один, собран из четырёх источников (папки в `dataset/`). Уровень аугментации для
каждого источника задан в `SOURCES` в `mri/config.py`: BDNeuro — без аугментации,
PMRAM, MultiClass и KaggleBrainTumor — уровень R.
Testing используется для итоговой оценки. Цель: precision и recall ≥ 0.85 по каждому классу.

Код — в пакете `mri/`; здесь только настройки запуска и результаты.

In [ ]:
import os

import cv2
import torch
from torch.utils.data import DataLoader

from mri.config import CLASSES, IMG_SIZE, SOURCES
from mri.model import simple_cnn
from mri.train import build_datasets, evaluate, fit, new_run_dir, predict, save_json, set_seed

EPOCHS = 30
EXPAND = 3       # для источников с аугментацией: сколько раз за эпоху сеть видит каждый снимок
BATCH = 64
NUM_WORKERS = int(os.environ.get("MRI_WORKERS", "4"))  # если DataLoader падает на Windows — поставьте 0
SEED = 0

set_seed(SEED)
torch.set_num_threads(4)
cv2.setNumThreads(1)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = device == "cuda"
RUN_DIR = new_run_dir()
print("Устройство:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
print("Аугментация:", SOURCES, "Результаты:", RUN_DIR, flush=True)

## Данные

Одна модель обучается на Training всех источников; в Testing аугментации нет.
Предобработка одинаковая: grayscale, обрезка полей, 224×224.

Результаты предварительные: между Training и Testing есть точные и почти одинаковые
снимки; очистка и разделение по пациентам не выполнены.

In [ ]:
train_data, test_data, test_table, counts = build_datasets(EXPAND)
train_dl = DataLoader(train_data, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS,
                      pin_memory=device == "cuda", persistent_workers=NUM_WORKERS > 0)
test_dl = DataLoader(test_data, batch_size=256, shuffle=False, num_workers=0, pin_memory=device == "cuda")

config = {"policies": SOURCES, "data_counts": counts, "epochs": EPOCHS, "batch": BATCH,
          "workers": NUM_WORKERS, "seed": SEED, "device": device, "torch": str(torch.__version__),
          "cuda": torch.version.cuda, "classes": list(CLASSES), "image_size": IMG_SIZE,
          "evaluation_note": "Preliminary: existing duplicate overlaps and patient separation unresolved."}
save_json(config, RUN_DIR / "config.json")
print("Снимков train:", sum(c["train_images"] for c in counts), "примеров за эпоху:", len(train_data),
      "снимков test:", len(test_data), flush=True)

## Модель
5 блоков «свёртка → нормализация → ReLU → уменьшение в 2 раза» (224 → 7), затем среднее по каждому каналу и линейный слой на 4 класса (`mri/model.py`).

In [ ]:
model = simple_cnn().to(device)
print(sum(p.numel() for p in model.parameters()), "параметров")

## Обучение
AdamW + OneCycle; после каждой эпохи `history.csv` и `model.pt` сохраняются в папку запуска.

In [ ]:
history = fit(model, train_dl, EPOCHS, device, RUN_DIR, config)

## Precision и recall на тесте
**Precision** — из снимков, которые модель отнесла к классу, доля верных.  
**Recall** — из снимков этого класса, доля найденных моделью.

In [ ]:
pred = predict(model, test_dl, device)
summary, by_class, confusion = evaluate(test_table, pred, RUN_DIR)
print("Итог на Testing (предварительная оценка, пересечения пока не устранены):")
print(summary.round(4).to_string(index=False))
print(by_class.round(4).to_string(index=False))
print("Сохранено:", RUN_DIR, flush=True)